# Bit Manipulation en C++

**Curso:** Programación Avanzada  
**Kernel:** xeus-cling (C++17)

---

Ya sabemos cómo se representan los números en binario. Ahora vamos a aprender a
**manipular esos bits directamente** desde C++, sin pasar por la aritmética normal.

- **`bitset`**: herramienta para *ver* los bits de un valor
- **Literales binarios y hexadecimales**: escribir valores en distintas bases en el código
- **Desplazamientos (`<<`, `>>`)**: mover todos los bits a la izquierda o derecha
- **Operadores lógicos bit a bit (`~`, `&`, `|`, `^`)**: operar bit a bit entre dos valores

La idea central: los enteros en memoria son simplemente secuencias de bits,
y C++ nos da herramientas para leerlos y modificarlos uno por uno.

---

## Requisitos previos

Ejecuta esta celda **antes de continuar**. Incluye tres bibliotecas:

| Biblioteca | Para qué la usamos |
|---|---|
| `<iostream>` | `cout` para imprimir resultados |
| `<bitset>` | `bitset<N>` para visualizar bits |
| `<iomanip>` | `setw(n)` para alinear columnas en la salida |

`setw(n)` (**set width**) le indica a `cout` que el siguiente valor debe ocupar
al menos `n` caracteres, rellenando con espacios por la izquierda si es necesario.
Lo usamos para que los números queden alineados junto a sus representaciones binarias
y la salida sea más fácil de leer:

```
sin setw:   1 << 0 = 1  ->  00000001
            1 << 4 = 16  ->  00010000   ← desalineado

con setw:   1 << 0 =   1  ->  00000001
            1 << 4 =  16  ->  00010000   ← alineado
```

In [ ]:
#include <iostream>
#include <bitset>
#include <iomanip>
using namespace std;

using bin8  = bitset<8>;   // alias para ver 8 bits
using bin32 = bitset<32>;  // alias para ver 32 bits

---

## Parte 1 — Ver los bits de un valor

### 1.1 `bitset`: imprimir en binario

`bitset<N>` convierte cualquier entero a su representación en `N` bits.
Es solo para *visualizar*; no cambia el valor almacenado.

In [2]:
{
    int x = 30;
    cout << "decimal: " << x        << endl;
    cout << "binario: " << bin8(x)  << endl;
}

decimal: 30
binario: 00011110


**¿Por qué `00011110`?**  
16 + 8 + 4 + 2 = 30. Los cuatro bits bajos encendidos corresponden exactamente a esas potencias de 2.

### 1.2 Literales en distintas bases

C++14 permite escribir un mismo valor en decimal, binario (`0b`) o hexadecimal (`0x`).
Para el compilador son idénticos; es solo una cuestión de legibilidad.

In [3]:
{
    int a = 30;          // decimal
    int b = 0b00011110;  // binario (C++14)  — el mismo número
    int c = 0x1E;        // hexadecimal      — el mismo número

    cout << "decimal: " << a << "  ->  " << bin8(a) << endl;
    cout << "binario: " << b << "  ->  " << bin8(b) << endl;
    cout << "hex:     " << c << "  ->  " << bin8(c) << endl;
}

decimal: 30  ->  00011110
binario: 30  ->  00011110
hex:     30  ->  00011110


### 1.3 Negativos: complemento a 2 en acción

Cuando usamos `bin32` en un negativo vemos todos los bits, incluido el patrón de complemento a 2.

In [4]:
{
    int z = -1;
    cout << "--- z = -1 ---" << endl;
    cout << "8  bits: " << bin8(z)  << endl;
    cout << "32 bits: " << bin32(z) << endl;
    cout << endl;

    z = -40;
    cout << "--- z = -40 ---" << endl;
    cout << "8  bits: " << bin8(z)  << endl;
    cout << "32 bits: " << bin32(z) << endl;
}

--- z = -1 ---
8  bits: 11111111
32 bits: 11111111111111111111111111111111

--- z = -40 ---
8  bits: 11011000
32 bits: 11111111111111111111111111011000


**`-1` siempre es todos 1s** sin importar cuántos bits uses — es la propiedad fundamental del complemento a 2.

**`-40`:** 40 = `00101000`, invertimos → `11010111`, sumamos 1 → `11011000`. Confirmado.

### 1.4 `char` también son bits

Un `char` es simplemente un entero de 8 bits. Podemos inicializarlo con cualquier literal.

In [5]:
{
    char c1 = 'a';         // desde un carácter: 'a' = 97
    char c2 = 0b01001101;  // desde binario literal → 77 = 'M'
    char c3 = 0x44;        // desde hexadecimal    → 68 = 'D'

    cout << "c1 = '" << c1 << "'  ->  " << bin8(c1) << endl;
    cout << "c2 = '" << c2 << "'  ->  " << bin8(c2) << endl;
    cout << "c3 = '" << c3 << "'  ->  " << bin8(c3) << endl;
}

c1 = 'a'  ->  01100001
c2 = 'M'  ->  01001101
c3 = 'D'  ->  01000100


---

## Parte 2 — Patrones especiales

Antes de ver los operadores, vale la pena conocer tres patrones que aparecen constantemente
en bit manipulation.

### 2.1 Apagar todos los bits: `x = 0`

In [6]:
{
    int x = 0;
    cout << "x = " << x << "  ->  " << bin32(x) << endl;
}

x = 0  ->  00000000000000000000000000000000


### 2.2 Encender todos los bits: `x = -1`

In [7]:
{
    int x = -1;
    cout << "x = " << x << "  ->  " << bin32(x) << endl;
}

x = -1  ->  11111111111111111111111111111111


**¿Por qué `-1` enciende todos los bits?**  
En complemento a 2, `-1 = 2^32 - 1 = 0xFFFFFFFF`. Todos los bits en 1.

### 2.3 Encender un bit específico: `1 << k`

El valor `1 << k` tiene exactamente el bit `k` encendido y todos los demás apagados.
Estas expresiones se llaman **máscaras de un bit**.

In [9]:
{
    for (int k = 0; k <= 7; k++) {
        int mascara = 1 << k;
        cout << "1 << " << k << " = "
             << setw(3) << mascara << "  ->  " << bin8(mascara) << endl;
    }
}

In file included from <<< inputs >>>:1:
input_line_11:5:17: error: use of undeclared identifier 'setw'; did you mean 'getw'?
    5 |              << setw(3) << mascara << "  ->  " << bin8(mascara) << endl;
      |                 ^~~~
      |                 getw
/usr/include/stdio.h:643:12: note: 'getw' declared here
  643 | extern int getw (FILE *__stream) __nonnull ((1));
      |            ^
In file included from <<< inputs >>>:1:
input_line_11:5:22: error: cannot initialize a parameter of type 'FILE *' (aka '_IO_FILE *') with
      an rvalue of type 'int'
    5 |              << setw(3) << mascara << "  ->  " << bin8(mascara) << endl;
      |                      ^
/usr/include/stdio.h:643:24: note: passing argument to parameter '__stream' here
  643 | extern int getw (FILE *__stream) __nonnull ((1));
      |                        ^
Failed to parse via ::process:Parsing failed.


Error: : Compilation error! In file included from <<< inputs >>>:1:
[1minput_line_11:5:17: [0m[0;1;31merror: [0m[1muse of undeclared identifier 'setw'; did you mean 'getw'?[0m
    5 |              << setw([0;32m3[0m) << mascara << [0;32m"  ->  "[0m << bin8(mascara) << endl;[0m
      | [0;1;32m                ^~~~
[0m      | [0;32m                getw
[0m[1m/usr/include/stdio.h:643:12: [0m[0;1;36mnote: [0m'getw' declared here[0m
  643 | [0;34mextern[0m [0;34mint[0m getw (FILE *__stream) __nonnull (([0;32m1[0m));[0m
      | [0;1;32m           ^
[0mIn file included from <<< inputs >>>:1:
[1minput_line_11:5:22: [0m[0;1;31merror: [0m[1mcannot initialize a parameter of type 'FILE *' (aka '_IO_FILE *') with
      an rvalue of type 'int'[0m
    5 |              << setw([0;32m3[0m) << mascara << [0;32m"  ->  "[0m << bin8(mascara) << endl;[0m
      | [0;1;32m                     ^
[0m[1m/usr/include/stdio.h:643:24: [0m[0;1;36mnote: [0mpassing argumen

Nota que `1 << k` es exactamente $2^k$. El desplazamiento a la izquierda
**multiplica por 2** con cada posición — ya lo veremos formalmente.

---

## Parte 3 — Desplazamiento izquierdo `<<`

`x << k` desplaza todos los bits de `x` hacia la izquierda `k` posiciones.
Los bits que salen por la izquierda se pierden; los huecos de la derecha se llenan con `0`.

```
  6 = 0 0 0 0 0 1 1 0
           ← ← ← ← ← ←  (desplazar 2)
  ?   0 0 0 1 1 0 0 0  = 24
```

**Efecto aritmético:** `x << k` equivale a multiplicar por $2^k$, siempre que no haya desbordamiento.

In [9]:
{
    int x = 6;
    cout << "x     = " << setw(2) << x      << "  ->  " << bin8(x)      << "   " << endl;
    cout << "x<<1  = " << setw(2) << (x<<1) << "  ->  " << bin8(x<<1)   << "   (" << x << " × 2)" << endl;
    cout << "x<<2  = " << setw(2) << (x<<2) << "  ->  " << bin8(x<<2)   << "   (" << x << " × 4)" << endl;
    cout << "x<<3  = " << setw(2) << (x<<3) << "  ->  " << bin8(x<<3)   << "   (" << x << " × 8)" << endl;
}

x     =  6  ->  00000110
x<<1  = 12  ->  00001100   (6 × 2)
x<<2  = 24  ->  00011000   (6 × 4)
x<<3  = 48  ->  00110000   (6 × 8)


---

## Parte 4 — Desplazamiento derecho `>>`

`x >> k` desplaza todos los bits hacia la derecha `k` posiciones.
Los bits que salen por la derecha se pierden.

**Efecto aritmético:** `x >> k` equivale a la **división entera** por $2^k$ (se trunca hacia abajo).

```
  100 = 0 1 1 0 0 1 0 0
              → → → →  (desplazar 2)
    ?   0 0 0 1 1 0 0 1  = 25    (100 / 4 = 25)
```

In [10]:
{
    int x = 100;
    cout << "x     = " << setw(3) << x      << "  ->  " << bin8(x)      << endl;
    cout << "x>>1  = " << setw(3) << (x>>1) << "  ->  " << bin8(x>>1)   << "   (100 / 2)" << endl;
    cout << "x>>2  = " << setw(3) << (x>>2) << "  ->  " << bin8(x>>2)   << "   (100 / 4)" << endl;
    cout << "x>>3  = " << setw(3) << (x>>3) << "  ->  " << bin8(x>>3)   << "   (100 / 8, truncado)" << endl;
}

x     = 100  ->  01100100
x>>1  =  50  ->  00110010   (100 / 2)
x>>2  =  25  ->  00011001   (100 / 4)
x>>3  =  12  ->  00001100   (100 / 8, truncado)


> **Nota:** `100 >> 3 = 12`, no 12.5. El desplazamiento siempre trunca hacia abajo,
> igual que la división entera `100 / 8 = 12` en C++.

---

## Parte 5 — NOT bit a bit `~`

`~x` invierte cada bit: los 0 se vuelven 1 y los 1 se vuelven 0.

| bit | `~bit` |
|:---:|:------:|
|  0  |   1    |
|  1  |   0    |

In [11]:
{
    int x = 30;
    cout << " x  = " << setw(3) <<  x << "  ->  " << bin32( x) << endl;
    cout << "~x  = " << setw(3) << ~x << "  ->  " << bin32(~x) << endl;
}

 x  =  30  ->  00000000000000000000000000011110
~x  = -31  ->  11111111111111111111111111100001


**¿Por qué `~30 = -31`?**  
`~x` en complemento a 2 equivale siempre a `-(x + 1)`.  
Así: `~30 = -(30 + 1) = -31`. Y en general, `~0 = -1`, `~(-1) = 0`.

---

## Parte 6 — AND bit a bit `&`

`a & b` opera cada par de bits de forma independiente:
el resultado es `1` solo si **ambos** bits son `1`.

| `a` | `b` | `a & b` |
|:---:|:---:|:-------:|
|  1  |  1  |    1    |
|  1  |  0  |    0    |
|  0  |  1  |    0    |
|  0  |  0  |    0    |

**Uso típico:** *apagar* bits específicos (aplicar una máscara).

In [12]:
{
    int a = 23, b = 35;
    cout << " a     = " << setw(2) << a     << "  ->  " << bin8(a)   << endl;
    cout << " b     = " << setw(2) << b     << "  ->  " << bin8(b)   << endl;
    cout << " a & b = " << setw(2) << (a&b) << "  ->  " << bin8(a&b) << endl;
}

 a     = 23  ->  00010111
 b     = 35  ->  00100011
 a & b =  3  ->  00000011


Solo los bits que están encendidos en **los dos** operandos a la vez sobreviven.
Aquí solo los bits 0 y 1 coinciden entre 23 y 35, por eso el resultado es `00000011 = 3`.

---

## Parte 7 — OR bit a bit `|`

`a | b` produce `1` si **al menos uno** de los dos bits es `1`.

| `a` | `b` | `a \| b` |
|:---:|:---:|:--------:|
|  1  |  1  |    1     |
|  1  |  0  |    1     |
|  0  |  1  |    1     |
|  0  |  0  |    0     |

**Uso típico:** *encender* bits específicos.

In [13]:
{
    int a = 23, b = 35;
    cout << " a     = " << setw(2) << a     << "  ->  " << bin8(a)   << endl;
    cout << " b     = " << setw(2) << b     << "  ->  " << bin8(b)   << endl;
    cout << " a | b = " << setw(2) << (a|b) << "  ->  " << bin8(a|b) << endl;
}

 a     = 23  ->  00010111
 b     = 35  ->  00100011
 a | b = 55  ->  00110111


El resultado recoge todos los bits encendidos en cualquiera de los dos operandos.

---

## Parte 8 — XOR bit a bit `^`

`a ^ b` produce `1` si los dos bits son **distintos** (eXclusive OR).

| `a` | `b` | `a ^ b` |
|:---:|:---:|:-------:|
|  1  |  1  |    0    |
|  1  |  0  |    1    |
|  0  |  1  |    1    |
|  0  |  0  |    0    |

**Uso típico:** *invertir* bits específicos. También aparece en criptografía y detección de errores.

In [14]:
{
    int a = 23, b = 35;
    cout << " a     = " << setw(2) << a     << "  ->  " << bin8(a)   << endl;
    cout << " b     = " << setw(2) << b     << "  ->  " << bin8(b)   << endl;
    cout << " a ^ b = " << setw(2) << (a^b) << "  ->  " << bin8(a^b) << endl;
}

 a     = 23  ->  00010111
 b     = 35  ->  00100011
 a ^ b = 52  ->  00110100


Los bits donde 23 y 35 **coinciden** se apagan; los que **difieren** se encienden.

> **Truco clásico con XOR:** `x ^ x = 0` siempre, y `x ^ 0 = x` siempre.
> Aplicar XOR dos veces con el mismo valor regresa al original:
> `(x ^ k) ^ k = x`. Base de muchos algoritmos de cifrado simple.

---

## Parte 9 — Combinando operadores

Los operadores de bits se pueden encadenar igual que la aritmética normal.
Veamos paso a paso la expresión `(~(13 >> 2)) & 39`:

In [15]:
{
    int x = 13, y = 39;
    cout << "x           = " << setw(3) << x           << "  ->  " << bin8(x)           << endl;
    cout << "x >> 2      = " << setw(3) << (x>>2)      << "  ->  " << bin8(x>>2)        << "   (descartamos los 2 bits bajos)" << endl;
    cout << "~(x >> 2)   = " << setw(3) << ~(x>>2)     << "  ->  " << bin32(~(x>>2))    << endl;
    cout << "y           = " << setw(3) << y            << "  ->  " << bin8(y)           << endl;
    cout << "~(x>>2) & y = " << setw(3) << (~(x>>2)&y) << "  ->  " << bin8(~(x>>2)&y)  << endl;
}

x           =  13  ->  00001101
x >> 2      =   3  ->  00000011   (descartamos los 2 bits bajos)
~(x >> 2)   =  -4  ->  11111111111111111111111111111100
y           =  39  ->  00100111
~(x>>2) & y =  36  ->  00100100


**Lectura del resultado paso a paso:**

1. `13 >> 2 = 3` → tiramos los 2 bits bajos de 13 (`01` se va por la derecha)
2. `~3` → invertimos todo: `00000011` se vuelve `11111111...11111100`
3. `& 39` → de todo ese patrón, solo sobreviven los bits que 39 tiene encendidos
   (`00100111`). El AND actúa como filtro.
4. Resultado: `00100100 = 36`

---

## Ejercicios

Resuelve **a mano** cada expresión, mostrando los bits intermedios.
Escribe el resultado final en decimal. Luego verifica con una celda de código.

### Ejercicio 1 — `17 & 12`

In [ ]:
// Tu solución aquí


### Ejercicio 2 — `7 << 2`

In [ ]:
// Tu solución aquí


### Ejercicio 3 — `13 | 12`

In [ ]:
// Tu solución aquí


### Ejercicio 4 — `(~7 | (1<<1) | (1<<3) | (1<<5)) & 29`

*Pista: calcula primero `~7`, luego OR con cada máscara, y al final aplica el AND.*

In [ ]:
// Tu solución aquí


### Ejercicio 5 — `(~(13 >> 2)) & 39`

*Este es el ejemplo que ya vimos paso a paso — intenta reproducirlo sin mirar.*

In [ ]:
// Tu solución aquí


---

## Resumen

| Operador | Sintaxis | Qué hace |
|----------|----------|----------|
| `bitset<N>` | `bin8(x)` | Muestra `x` en `N` bits (solo visualización) |
| Literal binario | `0b1010` | Escribe un valor directamente en binario (C++14) |
| Literal hex | `0x1F` | Escribe un valor en hexadecimal |
| Left shift | `x << k` | Desplaza bits a la izquierda; equivale a `x × 2^k` |
| Right shift | `x >> k` | Desplaza bits a la derecha; equivale a `x / 2^k` (entero) |
| NOT bit a bit | `~x` | Invierte todos los bits; equivale a `-(x+1)` |
| AND bit a bit | `x & y` | `1` solo si **ambos** bits son `1`; filtra/apaga bits |
| OR bit a bit | `x \| y` | `1` si **al menos uno** es `1`; enciende bits |
| XOR bit a bit | `x ^ y` | `1` si los bits son **distintos**; invierte bits |
| Máscara de un bit | `1 << k` | Solo el bit `k` encendido; vale `2^k` |

---
*Programación Avanzada — Universidad Panamericana*